# Gradient Boosted Decision Trees

Basically uses low accuracy decision trees and corrects them sequentially(improving the output of one tree with its child)

At each step, add a new model that fits the gradient of the loss function(i.e direction that minimizes error)

## Objective Function

$\Large{\mathcal{L} = \sum\limits_{i = 1}^{n}l(y_i, \hat{y}_i) + \sum\limits_{k = 1}^{K} \Omega(f_k))}$

Where,

- $l(y, \hat{y}_i)$ : Loss function (Sq. Loss or Log Loss)
- $\hat{y}_i$ : Prediction of the instance i
- $f_k$ : The $k_th$ tree function (This function maps input features to predictions)
- $\Omega(f)$ : Regularization function for complexity control

### Additive Model
The model is built additively as:

$\Large{\hat{y}_i^{(t)} = \hat{y}_i^{(t - 1)} + f_t(x_i)}$

Note that this is the next prediction, and $t - 1$ term is the current prediction.

So, the loss function to minimize becomes, $\Large{l(y_i, \hat{y}_i^{(t - 1)} + f_t(x_i))}$.

The loss function is complex, so we approximate it using simple polynomials, i.e using Taylor Series.

### Taylor Series Expansion

Now, let's approximate this complex function using 2nd Order Taylor expansion.

Since, we are expanding this function around $\hat{y}_i^{(t-1)}$. This is our $a$.

So $\Large{f(a) = f(y_i, \hat{y}_i^{(t-1)})}$

Remember that this is a multivariate function and $y_i$ is the label, so it remains constant.

Now we suppose that the first derivative is $g_i$ and second derivative is $h_i$.

**Note:**
**For our case: a = $\hat{y}_i^{(t-1)}$**

**And z = $\hat{y}_i^{(t - 1)} + f_t(x_i)$**

**In the expansion: $f(z) = f(a) + f^{'}(a)(z - a) + \frac{1}{2}f^{''}(a)(z-a)^2$**

**So, $z-a = f_t(x_i)$**

*With this in mind, the Taylor series approximation is:*

$\ell(y_i, \hat{y}_i^{(t-1)} + g_i f_t(x_i) + \frac{1}{2}h_i (f_t(x_i))^2)$


### Regression Tree
xgBoost models each $f_t$ as a regression tree:

$\Large{f_t(x_i) = \omega_{q(x_i)}}$

Where,

- $q(x_i) \in 1,2, \ldots , T$ assigns inputs $x_i$ to a leaf index

- $w_j$ is the prediction score(weight) of leaf $j$

- $T$ is the total number of leaves in the tree

### Regularization Term

$\Large{\Omega(f_t) = \gamma T + \frac{1}{2} \lambda \sum\limits_{j = 1}^{T} w_j^2}$

Where:

- $\Large{\gamma}$ penalizes the number of tree(controls the complexity of the tree)
- $\Large{\lambda}$ is the L2 regularization term on the leaf weights

### Putting it all together

For each leaf $j$, define the set of training examples assigned to it:

$\Large{I_j = \{i | q(x_i) = j\}}$

We summarize the gradients within each leaf:

- First order gradient sum:
> $G_j = \sum_{i \in I_j} g_i$
- Second order gradient sum:
> $H_j$ = $\sum_{i \in I_j}h_i$


i.e 

$\widetilde{\mathcal{L}}^{(t)}$ $\Large{ = \sum\limits_{j = 1}^{T} [G_j w_j + \frac{1}{2}H_j w_j ^ 2] + \gamma T + \frac{1}{2} \lambda \sum\limits_{j = 1}^{T} w_j^2}$

---
Grouping further:

$\Large{\widetilde{\mathcal{L}}^{(t)}$ = $\sum\limits_{j = 1}^{T}[G_j w_j + \frac{1}{2}(H_j + \lambda) w_j^2] + \gamma T}$

---
To minimize this, we take derivative w.r.t each $w_j$ and set to 0:

$\Large{\frac{d}{dw_j} (G_j w_j + \frac{1}{2}(H_j + \lambda) w_j^2) = 0}$

This gives:
$\Large{G_j + (H_j + \lambda)w_j = 0}$ => $\Large{w^*_j = -\frac{G_j}{H_j + \lambda}}$

---
Now, pluging $w^*_j$ back to the loss function

$\Large{\widetilde{\mathcal{L}}^{(t)}}$ = $\large{\frac{1}{2}\sum\limits_{j = 1}^{T}\frac{G_j^2}{H_j + \lambda} + \gamma T}$

>**This is the score of the whole tree, which XGBoost uses to decide how good a tree structure is**

## Splitting Trees

We now have a way to see how good a tree structure. The next step would be to see if the current tree can be split, so we can reduce the loss further. For this we get the formula for gain:

**Gain** = $\Large{\frac{1}{2} \left[\frac{G_L^2}{H_L+\lambda}+\frac{G_R^2}{H_R+\lambda}-\frac{(G_L+G_R)^2}{H_L+H_R+\lambda}\right] - \gamma}$

Understand this as:

1) The score on the new left leaf
2) The score on the new right leaf
3) The score on the original leaf
4) Regularization on the additional leaf

If Gain $\gt 0$, split is beneficial, else stop splitting

# Programming Steps

- **Step 0**
    > For regression, set the initial prediction to mean value of the labels, i.e $\hat{y}_i^{(0)} = mean(y)$
    > 
    > For classification, set the initial classification to $\hat{y}_i^{(0)} = $ $log\frac{p}{1-p}$ where, $p = \frac{1}{n}\sum{y_i}$
- **Step 1**
    > For each boosting round $t=1$ to $T$
    >
    > Compute Gradients and Hessians i.e for each sample $i$ Compute $g_i$ and $h_i$
    >
    > Build the decision tree $f_t(x)$ using Gradients and Hessians.
    >
    > Start at root node with all samples.
    >
    > At each split, compute the gain with the split gain formula and watch for stopping criteria
    >
    > Assign weight to each leaf. For each leaf, $w_j$ = $-\frac{G_j}{H_j + \lambda}$
    >
    > Update predictions for each sample, $\hat{y}_i^{t} = \hat{y}^{(t-1)}_i + n \cdot f_t(x_i)$
- **Step 2**
    > For regression, Output answer as: $\hat{y}_i^{(T)}$
    >
    > For classification, apply sigmoid to $y_i^{T}$ to get $\hat{p}_i$.
    > Class = 1 for $\hat{p}_i \gt 0.5$, else 0

In [2]:
!pip install xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.9/253.9 MB 9.5 MB/s eta 0:00:00m eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.4/322.4 MB 7.2 MB/s eta 0:00:00m eta 0:00:010:00:02m

[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [14]:
import xgboost as xgb
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error
import numpy as np

In [5]:
X, y = make_regression(n_samples=10000, n_features = 15, noise = 14.89, random_state = 69)

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 59)

In [8]:
model = xgb.XGBRegressor(objective='reg:squarederror', eval_metric='rmse')
model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric='rmse', feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=None, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=None,
             n_jobs=None, num_parallel_tree=None, ...)

In [10]:
y_pred = model.predict(X_test)

In [11]:
mse = mean_squared_error(y_test, y_pred)
print(f"Accuracy: {mse}")

Accuracy: 1567.278194237137


In [12]:
print(y.mean(), y.std())

0.5635225463014311 180.43038004455062


In [18]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(rmse)

39.58886452321078
